In [1]:
pip install boto3

DEPRECATION: Loading egg at /opt/homebrew/lib/python3.12/site-packages/cvxopt-0.0.0-py3.12-macosx-14.0-arm64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import json
import csv
import os
from datetime import datetime
import boto3
from botocore.exceptions import NoCredentialsError
from config import AWS_ACCESS_KEY, AWS_SECRET_KEY, AWS_REGION, BUCKET_NAME

In [9]:
API_KEY = ''  # Get from https://www.eia.gov/opendata/register.php
BASE_URL = 'https://api.eia.gov/v2/crude-oil-imports/data/'

In [4]:
datasets = {

    "petroleum": {
        "url": "https://api.eia.gov/v2/petroleum/pnp/wprodrb/data/",
        "frequency": "weekly",
        "data": ["value"],
        "start": "2026-03-01",
        "end": "2026-03-06"
    },

    "natural_gas": {
        "url": "https://api.eia.gov/v2/natural-gas/prod/whv/data/",
        "frequency": "monthly",
        "data": ["value"],
        "start": "2025-12",
        "end": "2025-12"
    },

    "electricity": {
        "url": "https://api.eia.gov/v2/electricity/electric-power-operational-data/data/",
        "frequency": "monthly",
        "data": ["consumption-for-eg", "consumption-for-eg-btu"],
        "start": "2025-12",
        "end": "2025-12"
    },

    "crude_oil_imports": {
        "url": "https://api.eia.gov/v2/crude-oil-imports/data/",
        "frequency": "monthly",
        "data": ["quantity"],
        "start": "2025-12",
        "end": "2025-12"
    }
}

In [5]:
BASE_OUTPUT = "data/raw"

os.makedirs(BASE_OUTPUT, exist_ok=True)

s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

In [6]:
def fetch_dataset(name, config):

    print(f"\nFetching dataset: {name}")

    output_folder = os.path.join(BASE_OUTPUT, name, config["frequency"])
    os.makedirs(output_folder, exist_ok=True)

    params = {
        "api_key": API_KEY,
        "frequency": config["frequency"],
        "start": config["start"],
        "end": config["end"],
        "offset": 0,
        "length": 5000
    }

    for i, d in enumerate(config["data"]):
        params[f"data[{i}]"] = d

    params["sort[0][column]"] = "period"
    params["sort[0][direction]"] = "desc"

    headers = {
        "X-Params": json.dumps({
            "frequency": config["frequency"],
            "data": config["data"],
            "facets": {},
            "start": config["start"],
            "end": config["end"],
            "sort": [{"column": "period", "direction": "desc"}],
            "offset": 0,
            "length": 5000
        })
    }

    all_records = []
    offset = 0
    total = None

    while True:

        params["offset"] = offset

        response = requests.get(config["url"], params=params, headers=headers)

        if response.status_code != 200:
            print("API error:", response.text)
            break

        data = response.json()

        records = data.get("response", {}).get("data", [])
        total = int(data.get("response", {}).get("total", 0))

        if not records:
            break

        all_records.extend(records)

        pct = (len(all_records) / total * 100) if total else 0

        print(f"{name}: {len(all_records):,} / {total:,} ({pct:.1f}%)")

        if len(all_records) >= total:
            break

        offset += 5000

    if not all_records:
        print("No data retrieved.")
        return

    today = datetime.today().strftime("%Y%m%d")

    json_file = os.path.join(output_folder, f"{name}_{today}.json")

    with open(json_file, "w") as f:
        json.dump(all_records, f, indent=2)

    csv_file = os.path.join(output_folder, f"{name}_{today}.csv")

    with open(csv_file, "w", newline="") as f:

        writer = csv.DictWriter(f, fieldnames=all_records[0].keys())

        writer.writeheader()

        writer.writerows(all_records)

    print(f"Saved CSV: {csv_file}")

    upload_to_s3(csv_file, name, config["frequency"])


In [7]:
def upload_to_s3(file_path, dataset, frequency):

    s3_key = f"raw/{dataset}/{frequency}/{os.path.basename(file_path)}"

    try:

        s3_client.upload_file(file_path, BUCKET_NAME, s3_key)

        print(f"Uploaded to s3://{BUCKET_NAME}/{s3_key}")

    except Exception as e:

        print("S3 upload failed:", str(e))

In [8]:
for name, config in datasets.items():

    fetch_dataset(name, config)

print("\nPipeline complete.")


Fetching dataset: petroleum
petroleum: 109 / 109 (100.0%)
Saved CSV: data/raw/petroleum/weekly/petroleum_20260311.csv
Uploaded to s3://bigdata-stocks/raw/petroleum/weekly/petroleum_20260311.csv

Fetching dataset: natural_gas
natural_gas: 37 / 37 (100.0%)
Saved CSV: data/raw/natural_gas/monthly/natural_gas_20260311.csv
Uploaded to s3://bigdata-stocks/raw/natural_gas/monthly/natural_gas_20260311.csv

Fetching dataset: electricity
electricity: 5,000 / 17,456 (28.6%)
electricity: 10,000 / 17,456 (57.3%)
electricity: 15,000 / 17,456 (85.9%)
electricity: 20,000 / 17,456 (114.6%)
Saved CSV: data/raw/electricity/monthly/electricity_20260311.csv
Uploaded to s3://bigdata-stocks/raw/electricity/monthly/electricity_20260311.csv

Fetching dataset: crude_oil_imports
crude_oil_imports: 2,091 / 2,091 (100.0%)
Saved CSV: data/raw/crude_oil_imports/monthly/crude_oil_imports_20260311.csv
Uploaded to s3://bigdata-stocks/raw/crude_oil_imports/monthly/crude_oil_imports_20260311.csv

Pipeline complete.
